# Vle / StudentVle 분석

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
# 폰트 테마 설정
sns.set_theme(style="whitegrid")
# 한글 폰트 설정 (맑은 고딕) — set_theme 이후에 지정해야 덮어쓰이지 않음
plt.rc("font", family="Malgun Gothic")
# 마이너스 기호 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

# 1. 데이터 불러오기

In [ ]:
vle = pd.read_csv("../CSV_files/vle.csv")
st_vle = pd.read_csv("../CSV_files/studentVle.csv")

print(vle.head())
print(st_vle.head())

In [ ]:
print(vle.info())
print(st_vle.info())

In [ ]:
print(vle.describe())
print(st_vle.describe())

In [ ]:
print(vle.isnull().sum())
print(st_vle.isnull().sum())

## 1) vle.csv : 온라인 강의실의 교재 목록 (카탈로그)

- id_site (자료 고유 번호): 강의실 자료 '바코드 번호'

- code_module / code_presentation: 과목 코드(예: AAA)와 학기 정보(예: 2013J)  

- activity_type (자료 유형) : 어떤 종류의 공부 자료인지 나타내는 성격이에요. 

    예를 들어 forum은 토론 게시판, quiz는 문제 풀이, homepage는 메인 화면, subpage는 보조 학습 페이지
    
    학생들이 어떤 페이지를 주로 보다가 지쳐서 나갔는지 분석할 수 있는 엄청난 무기입니다.  

- week_from / week_to (권장 학습 시작/종료 주차): 교수님이 추천해 둔 스케줄 가이드라인

## 2) studentVle.csv : 수강생들의 강의실 클릭 출석부

학생들이 매일 강의실에 들어와서 어떤 자료를 몇 번씩 클릭했는지 실시간으로 찍힌 '디지털 로그' 장부 

- code_module / code_presentation / id_student: 어떤 학생이 어떤 과목, 어떤 학기 반에서 활동했는지 알려주는 기본 인구통계 이름표.  

- id_site (클릭한 자료 번호): 학생이 강의실에서 마우스로 클릭한 공부 자료의 바코드 번호예요. 위의 vle.csv 장부와 연결해서 과목 성격을 알아낼 수 있습니다.

- date (활동일): 학기 시작일을 기준으로 몇 일째 되는 날에 접속했는지를 나타내요. 예를 들어 10이면 개강 후 10일째 되는 날에 공부했다는 뜻

- sum_click (하루 클릭 횟수): 그날 그 자료를 몇 번이나 마우스로 꾹꾹 눌러서 열어봤는지 기록한 '하루 공부량'

# 2. label_churn 정의 (이전 노트북과 동일한 기준)

01~03번 노트북과 동일하게, studentInfo.csv를 불러와 같은 규칙(Withdrawn=이탈, 나머지=완수)으로
label_churn을 만들어둡니다.

In [ ]:
info = pd.read_csv("../CSV_files/studentInfo.csv")

info["label_churn"] = info["final_result"].apply(lambda x: "이탈" if x == "Withdrawn" else "완수")

# 이전 노트북들과 동일한 값(완수 0.688 / 이탈 0.312)이 나와야 정상입니다.
print(info["label_churn"].value_counts(normalize=True).round(3))

# 3. 데이터 품질 검증

이 파일들은 규모가 매우 큽니다(studentVle.csv 1,065만 행). 큰 데이터일수록 결측치나 중복,
이상치를 미리 확인하지 않으면 파생 변수 계산 단계에서 결과가 통째로 왜곡될 수 있으므로,
이번에도 03번 노트북과 같은 방식으로 꼼꼼히 짚고 넘어갑니다.

### 3-1. vle.csv의 week_from / week_to 결측 확인

5,243건(전체 6,364건 중 82.4%)이나 결측인데, 이게 무작위 결측인지 아니면 특정 자료 유형의
구조적 특징인지 activity_type별로 확인합니다.

In [ ]:
print("week_from 결측 5,243건의 activity_type 분포(상위 항목):")
print(vle[vle["week_from"].isna()]["activity_type"].value_counts())

print("\nactivity_type별 결측 비율(%) - 높은 순:")
miss_ratio = vle.groupby("activity_type")["week_from"].apply(lambda x: x.isna().mean() * 100).round(1)
print(miss_ratio.sort_values(ascending=False))

**[확인 결과]** forumng(토론게시판), homepage(메인화면), glossary(용어집), oucollaborate,
ouelluminate, htmlactivity, sharedsubpage, folder, externalquiz는 **결측률이 정확히 100%**
입니다. 이 자료 유형들은 특정 주차에만 열리는 게 아니라 **학기 내내 상시로 열려있는 자료**라서
애초에 "권장 주차"라는 개념 자체가 없는 것으로 보입니다. 즉 이 결측은 데이터 오류가 아니라
**구조적으로 당연한 결측(자료 성격상 주차 지정이 필요 없음)**입니다.

반면 resource(94.4%), quiz(89.8%), oucontent(61.7%) 등은 주차가 지정된 경우와 안 된 경우가
섞여 있어서, 일부만 특정 주차에 국한된 자료로 보입니다. week_from/week_to는 이후 분석에서
필수 항목이 아니므로(활동 자체는 date로 이미 기록됨), 결측을 메우지 않고 그대로 둡니다.

### 3-2. studentVle.csv의 date 값 검증 (음수값 · 이상치)

date가 음수인 경우(개강 전 사전 접속)의 비율과, courses.csv의 module_presentation_length를
초과하는(=학기가 끝난 뒤에도 기록된) 이상치가 있는지 확인합니다.

In [ ]:
courses = pd.read_csv("../CSV_files/courses.csv")

neg_date = st_vle[st_vle["date"] < 0]
print("date < 0 (개강 전 사전 접속) 비율:", round(len(neg_date) / len(st_vle) * 100, 2), "%")
print(neg_date["date"].describe())

merged_len = pd.merge(st_vle, courses, on=["code_module", "code_presentation"], how="left")
over = merged_len[merged_len["date"] > merged_len["module_presentation_length"]]
print("\n학기 길이(module_presentation_length)를 초과해서 기록된 건수:", len(over), "/", len(merged_len))

**[확인 결과]** 개강 전 사전 접속은 전체의 6.47%(최대 25일 전, 평균 9일 전)로, 02번 노트북에서
본 사전 등록 패턴과 비슷하게 자연스러운 현상입니다. 학기 길이를 초과하는 기록은 **0건**으로,
date 값 자체는 깨끗합니다.

### 3-3. 조인 안전성 검증 (참조 무결성 · 키 일치 · label_churn 매칭)

이후 vle.csv, studentInfo.csv와 계속 조인해서 써야 하므로, 조인 전에 무결성을 확인합니다.

In [ ]:
# 1) studentVle의 id_site가 vle.csv에 전부 존재하는지
missing_sites = set(st_vle["id_site"]) - set(vle["id_site"])
print("studentVle에만 있고 vle엔 없는 id_site 수:", len(missing_sites))

# 2) 같은 id_site가 vle.csv와 studentVle.csv에서 항상 같은 code_module/code_presentation을 가리키는지
chk = pd.merge(
    st_vle[["id_site", "code_module", "code_presentation"]].drop_duplicates(),
    vle[["id_site", "code_module", "code_presentation"]],
    on="id_site", how="left", suffixes=("_sv", "_v")
)
mismatch = chk[(chk["code_module_sv"] != chk["code_module_v"]) | (chk["code_presentation_sv"] != chk["code_presentation_v"])]
print("id_site 기준 과목/학기 불일치 건수:", len(mismatch), "/", len(chk))

# 3) studentVle의 (학생, 과목, 학기) 조합이 studentInfo에 전부 존재하는지 (label_churn 조인 안전성)
key_check = st_vle[["id_student", "code_module", "code_presentation"]].drop_duplicates()
merged_info = pd.merge(key_check, info[["id_student", "code_module", "code_presentation", "label_churn"]],
                        on=["id_student", "code_module", "code_presentation"], how="left")
print("studentInfo와 매칭 안 되는 학생-과목 조합 수:", merged_info["label_churn"].isna().sum(), "/", len(merged_info))

**[확인 결과]** 세 가지 모두 **문제 0건**입니다. id_site 참조 무결성, 과목/학기 일치, label_churn
조인까지 전부 안전하게 확인되어 뒤에서 마음 놓고 조인해서 쓸 수 있습니다.

### 3-4. 중복 행 확인 — 핵심 발견

(id_student, code_module, code_presentation, id_site, date)가 동일한 행이 있는지 확인합니다.
이론적으로는 "그날 그 자료의 총 클릭수"가 한 행에 다 담겨 있어야 할 것 같은데, 실제로 그런지
봅니다.

In [ ]:
key = ["id_student", "code_module", "code_presentation", "id_site", "date"]
dup_mask = st_vle.duplicated(subset=key, keep=False)
print("동일 (학생,과목,학기,자료,날짜) 키를 공유하는 행 수:", dup_mask.sum(), "/", len(st_vle),
      f"({round(dup_mask.mean()*100,1)}%)")

# 다섯 개 키뿐 아니라 sum_click까지 완전히 동일한 '진짜' 중복 행
full_dup = st_vle.duplicated(keep=False)
print("모든 컬럼(값)까지 완전히 동일한 행 수:", full_dup.sum())

# 같은 키인데 sum_click 값이 서로 다른 경우가 있는지 (=같은 날 여러 번 접속을 나눠 기록한 것으로 추정)
grp_nunique = st_vle[dup_mask].groupby(key)["sum_click"].nunique()
print("같은 키에서 sum_click 값이 서로 다른 그룹 수:", (grp_nunique > 1).sum(), "/", len(grp_nunique))

**[확인 결과 — 매우 중요]** 같은 (학생,자료,날짜) 키를 공유하는 행이 **381만 건(전체의 35.8%)**
이나 됩니다. 이 중 73%(그룹 기준)는 **sum_click 값이 서로 다릅니다** — 즉 하나의 날짜/자료 조합에
대해 여러 번의 접속(세션)이 각각 별도 행으로 기록되어 있다는 뜻입니다(예: 오전에 2번 클릭,
오후에 다시 4번 클릭 → 두 행으로 따로 기록).

**[결론 — 이후 분석에 반드시 반영할 사항]** studentVle.csv는 "학생-자료-날짜당 1행"이 아니라
**세션 단위 로그**입니다. 따라서 "하루 총 클릭수"나 "학생별 총 클릭수" 같은 파생 변수를 만들 때는
행 개수를 세면 안 되고, **반드시 sum_click을 합산(sum)**해야 합니다. 행 단위로 세면 같은 방문을
중복 집계하거나 활동량을 실제보다 부풀리게 됩니다.

### 3-5. sum_click 분포 확인 (극단치 참고용)

이후 평균/합계를 계산할 때 극단치의 영향을 가늠할 수 있도록 분포만 간단히 확인해둡니다.

In [ ]:
print(st_vle["sum_click"].describe())
print("\n상위 1% 임계값:", st_vle["sum_click"].quantile(0.99))
print("sum_click > 100인 행 수:", (st_vle["sum_click"] > 100).sum(), "/", len(st_vle))

**[확인 결과]** 중앙값은 2인데 최댓값은 6,977로, 분포가 심하게 오른쪽으로 치우쳐 있습니다
(상위 1% 임계값이 34). 100건이 넘는 행도 6,757건 있습니다. 평균만 보면 소수의 극단치에
쉽게 왜곡될 수 있으므로, 이후 완수/이탈 그룹을 비교할 때는 **평균과 중앙값을 함께** 보고,
필요하면 로그 변환도 고려합니다.

### 3-6. 완전성 검증 (id_site 중복 · 과목 커버리지 · registration 대비 고아 레코드)

조인 안전성(3-3)에 이어, vle.csv 자체의 키 무결성과 세 파일 간 과목/학기 커버리지가
일치하는지, 그리고 studentRegistration.csv에 없는 학생-과목 조합이 studentVle.csv에
섞여 있지는 않은지 추가로 확인합니다.

In [ ]:
# 1) vle.csv에서 id_site가 유일한 값인지 (중복되면 조인 시 행이 불어나는 문제가 생길 수 있음)
print("vle.csv에서 id_site 중복 행 수:", vle.duplicated(subset=["id_site"]).sum(), "/", len(vle))

# 2) vle / studentVle / courses 세 파일의 (code_module, code_presentation) 커버리지가 일치하는지
vle_courses = set(zip(vle["code_module"], vle["code_presentation"]))
sv_courses = set(zip(st_vle["code_module"], st_vle["code_presentation"]))
courses_courses = set(zip(courses["code_module"], courses["code_presentation"]))
print("\ncourses.csv 과목 수:", len(courses_courses), "/ vle.csv 과목 수:", len(vle_courses),
      "/ studentVle.csv 과목 수:", len(sv_courses))
print("vle.csv에만 있고 studentVle엔 없는 과목:", vle_courses - sv_courses)
print("studentVle에만 있고 vle.csv엔 없는 과목:", sv_courses - vle_courses)

# 3) studentRegistration.csv에 없는 (학생,과목,학기) 조합이 studentVle.csv에 있는지 (등록 안 한 학생의 로그가 섞여있으면 안 됨)
reg = pd.read_csv("../CSV_files/studentRegistration.csv")
reg_keys = set(zip(reg["id_student"], reg["code_module"], reg["code_presentation"]))
sv_keys = set(zip(st_vle["id_student"], st_vle["code_module"], st_vle["code_presentation"]))
print("\nstudentVle에만 있고 studentRegistration엔 없는 학생-과목 조합 수:", len(sv_keys - reg_keys))

**[확인 결과]** 세 가지 모두 문제없습니다. id_site 중복 0건, 과목/학기 커버리지는 courses.csv
기준 22개 전부 vle.csv·studentVle.csv에 정확히 일치, studentRegistration.csv에 등록 기록이
없는데 활동 로그만 있는 "고아 레코드"도 0건입니다.

### 3-7. 탈퇴 이후에도 VLE 활동 기록이 남아있는지 확인 — 03번과 같은 종류의 편향 점검

03번 노트북에서 "이탈 학생의 미제출 과제 중 78.4%가 탈퇴일 이후 마감"이라는 편향을 발견했던
것과 같은 문제가 여기에도 있을 수 있습니다. 즉 이탈(Withdrawn) 학생의 date_unregistration
이후에도 VLE 로그가 찍혀 있다면, "활동 기간"이나 "마지막 접속일" 같은 파생 변수를 만들 때
왜곡이 생길 수 있으므로 미리 확인합니다.

In [ ]:
wd = info[info["label_churn"] == "이탈"][["id_student", "code_module", "code_presentation"]]
wd = pd.merge(wd, reg[["id_student", "code_module", "code_presentation", "date_unregistration"]],
              on=["id_student", "code_module", "code_presentation"], how="left")
wd_valid = wd.dropna(subset=["date_unregistration"])

sv_wd = pd.merge(st_vle, wd_valid, on=["id_student", "code_module", "code_presentation"], how="inner")
after = sv_wd[sv_wd["date"] > sv_wd["date_unregistration"]]

print("이탈 학생의 VLE 로그 중 탈퇴일 이후에 기록된 건수:", len(after), "/", len(sv_wd),
      f"({round(len(after) / len(sv_wd) * 100, 2)}%)")

n_affected = after[["id_student", "code_module", "code_presentation"]].drop_duplicates().shape[0]
n_total = sv_wd[["id_student", "code_module", "code_presentation"]].drop_duplicates().shape[0]
print("탈퇴 이후 로그가 있는 학생-과목 조합 수:", n_affected, "/", n_total,
      f"({round(n_affected / n_total * 100, 1)}%)")

gap = after["date"] - after["date_unregistration"]
print("\n탈퇴일 대비 초과 일수 분포(양수=탈퇴 이후 며칠 뒤 기록):")
print(gap.describe())

**[확인 결과]** 이탈 학생의 VLE 로그 중 **3.05%(29,050건)**가 탈퇴일 이후에 기록되어 있고,
이런 학생-과목 조합은 전체의 **17.9%(1,272 / 7,094)**에 달합니다. 탈퇴 이후 최대 423일이
지난 뒤에도 로그가 남아있는 경우가 있어서, date_unregistration이 VLE 접근을 막는 하드 컷오프는
아닌 것으로 보입니다(행정상 탈퇴 처리일과 실제 마지막 접속일 사이에 시차가 있을 수 있습니다).

03번의 78.4%에 비하면 비중은 훨씬 작지만(3.05%), **완전히 무시할 수준은 아닙니다.** 이후
"총 활동일수", "마지막 접속일", "학습 지속기간" 같은 파생 변수를 만들 때는 03번에서 세운 원칙과
동일하게 **탈퇴일 이후 로그는 제외**하고 계산하는 것이 안전합니다.

**[3번 전체 요약]**
1. week_from/week_to 결측(82%)은 상시자료 유형이라 구조적으로 정상.
2. date 값은 이상치 없이 깨끗함(개강 전 사전 접속 6.47%는 자연스러운 현상).
3. 조인 안전성(id_site 참조무결성·과목매칭·label_churn) 및 완전성(id_site 중복·과목 커버리지·
   registration 고아 레코드) 모두 이상 없음.
4. **세션 단위 중복 키(35.8%)** → 파생 변수는 반드시 groupby + sum으로 집계.
5. sum_click은 극단치가 커서(최댓값 6,977) 평균과 중앙값을 함께 볼 것.
6. **탈퇴일 이후 로그(3.05%, 학생 기준 17.9%)** → 활동 기간 관련 파생 변수는 탈퇴일 이전
   데이터만 사용해서 계산할 것.

# 4. 학생별 참여도 파생 변수 생성

3번에서 확정한 두 원칙(세션 단위 데이터는 groupby+sum으로 집계 / 이탈 학생은 탈퇴일 이후
로그를 제외)을 적용해서, 학생별 VLE 참여도 지표를 만듭니다.

### 4-1. 전처리: 이탈 학생의 탈퇴일 이후 로그 제거

3-7에서 확인한 대로, 이탈 학생인데 date_unregistration 이후에 기록된 로그(3.05%, 29,050건)는
제거합니다. 완수 학생과 date_unregistration이 없는 이탈 학생(93명, 판단 불가)은 원본을 그대로
둡니다.

In [ ]:
reg_small = reg[["id_student", "code_module", "code_presentation", "date_unregistration"]]

sv = pd.merge(st_vle, info[["id_student", "code_module", "code_presentation", "label_churn"]],
              on=["id_student", "code_module", "code_presentation"], how="left")
sv = pd.merge(sv, reg_small, on=["id_student", "code_module", "code_presentation"], how="left")

# 이탈 학생 + date_unregistration 존재 + 탈퇴일보다 나중에 기록된 로그만 제거 대상으로 표시
mask_exclude = (
    (sv["label_churn"] == "이탈")
    & (sv["date_unregistration"].notna())
    & (sv["date"] > sv["date_unregistration"])
)
print("제거되는 행 수:", mask_exclude.sum(), "/", len(sv))

sv_clean = sv[~mask_exclude].copy()

### 4-2. 학생별 참여도 지표 계산

정제된 데이터(sv_clean)를 학생-과목 단위로 묶어서, 다음 네 가지 지표를 만듭니다.

- total_click: 전체 클릭수 합계 (행 개수가 아니라 sum_click을 합산 — 3-4의 원칙)
- n_days_active: 실제로 접속한 날짜 수(고유 date 개수)
- first_active_day / last_active_day: 첫 접속일 / 마지막 접속일
- active_span: 마지막 접속일 - 첫 접속일 (학습이 이어진 기간)

In [ ]:
engagement = sv_clean.groupby(["id_student", "code_module", "code_presentation"]).agg(
    total_click=("sum_click", "sum"),
    n_days_active=("date", "nunique"),
    first_active_day=("date", "min"),
    last_active_day=("date", "max"),
).reset_index()

engagement["active_span"] = engagement["last_active_day"] - engagement["first_active_day"]

print("참여도 지표가 계산된 학생-과목 조합 수:", len(engagement))
print(engagement.describe().round(1))

### 4-3. 전체 학생 roster와 결합 (VLE 활동이 아예 없는 학생 포함)

위 engagement 테이블에는 VLE에 한 번이라도 접속한 학생만 들어 있습니다. 03번의 과제 제출률
분석과 마찬가지로, **접속 기록이 아예 없는 학생도 0건으로 포함**시켜야 완수/이탈 그룹을 공정하게
비교할 수 있으므로, studentInfo 기준 전체 학생-과목 roster와 left join합니다.

In [ ]:
roster = info[["id_student", "code_module", "code_presentation", "label_churn"]].drop_duplicates()

full_engagement = pd.merge(roster, engagement, on=["id_student", "code_module", "code_presentation"], how="left")

# 접속 기록이 아예 없으면 total_click/n_days_active는 0, first/last_active_day는 NaN(접속 자체가 없었으므로)
full_engagement["total_click"] = full_engagement["total_click"].fillna(0)
full_engagement["n_days_active"] = full_engagement["n_days_active"].fillna(0)

print("전체 학생-과목 조합 수:", len(full_engagement))
print("VLE 활동이 아예 없는 학생-과목 조합 수:", (full_engagement["n_days_active"] == 0).sum())

print("\n완수/이탈 그룹별 요약(참고용 미리보기 — 자세한 비교는 다음 단계에서 진행):")
print(full_engagement.groupby("label_churn")[["total_click", "n_days_active"]].agg(["mean", "median"]).round(1))

### 4-4. 파생 변수 검증 — 조인 안전성 및 극단치 재점검

파생 변수를 다음 단계(그룹 비교)에서 바로 쓰기 전에, 중간 조인 과정에서 행이 부풀려지지
않았는지, roster와 정확히 맞아떨어지는지, 그리고 이상해 보이는 극단치가 실제로 오류인지를
한 번 더 검증합니다.

In [ ]:
# 1) info / studentRegistration 키 유일성 재확인 (중복이 있으면 merge 시 행이 불어남)
print("info 키 중복:", info.duplicated(subset=["id_student", "code_module", "code_presentation"]).sum())
print("reg 키 중복:", reg.duplicated(subset=["id_student", "code_module", "code_presentation"]).sum())

# 2) 4-1에서 두 번 merge한 뒤에도 원본 행 수(1,065만)가 그대로 유지되는지
print("\nst_vle 원본 행 수:", len(st_vle), "/ merge 후 sv 행 수:", len(sv))
print("label_churn 매칭 안 된 행 수:", sv["label_churn"].isna().sum())

# 3) roster 행 수와 full_engagement 행 수가 정확히 같은지 (누락/중복 없이 1:1 매칭됐는지)
print("\nroster 행 수:", len(roster), "/ full_engagement 행 수:", len(full_engagement))

# 4) active_span이 음수인 경우(로직 오류)가 있는지, 활동 0인데 active_span만 남아있는 경우가 있는지
print("\n음수 active_span 개수:", (full_engagement["active_span"] < 0).sum())
zero_mask = full_engagement["n_days_active"] == 0
print("활동 0인데 active_span이 NaN이 아닌 경우:", (~full_engagement.loc[zero_mask, "active_span"].isna()).sum())

# 5) n_days_active가 학기 길이(module_presentation_length)보다 큰, 물리적으로 이상해 보이는 경우
chk_len = pd.merge(full_engagement, courses, on=["code_module", "code_presentation"], how="left")
weird = chk_len[chk_len["n_days_active"] > chk_len["module_presentation_length"]]
print("\nn_days_active > 학기 길이인 경우:", len(weird))
print("이 중 개강 전(first_active_day<0) 접속이 원인인 비율:",
      round((weird["first_active_day"] < 0).mean() * 100, 1), "%")

**[확인 결과]** 조인 관련 항목은 전부 정상입니다. info/reg 키 중복 0건, merge 후에도 원본
1,065만 행이 그대로 유지되고 label_churn 미매칭도 0건, roster와 full_engagement 행 수(32,593)도
정확히 일치합니다. active_span 음수나 "활동 0인데 span만 남아있는" 로직 오류도 없습니다.

다만 **n_days_active가 학기 길이를 초과하는 57건**을 발견했습니다. 처음엔 이상치로 의심했지만
확인해보니 **100% 개강 전(음수 날짜) 접속이 포함된 경우**였습니다 — 예를 들어 학기 시작 25일
전부터 접속해서 학기 마지막 날까지 꾸준히 활동한 학생은, 활동 범위가 "공식 학기 길이"보다
길어 보이는 게 당연합니다. 즉 **오류가 아니라 3-2에서 이미 확인한 사전 접속 현상의 자연스러운
결과**이므로 별도 처리 없이 그대로 둡니다.

total_click 최댓값(24,139)도 확인해보니 실제로 270일 이상 꾸준히 접속한 최상위 참여 학생들의
값으로, 데이터 오류가 아니라 정상적인 극단치입니다.

**[결론]** 파생 변수(total_click, n_days_active, first/last_active_day, active_span)는 검증을
통과했고, 다음 단계인 완수/이탈 그룹 비교에 그대로 사용해도 안전합니다.

# 5. 완수/이탈 그룹별 VLE 참여도 비교

4번에서 만든 참여도 지표(total_click, n_days_active, 활동 여부, active_span)를 완수/이탈
그룹으로 나눠서 비교합니다. 03번의 과제 제출률 분석과 같은 흐름으로, 클릭수·접속일수·
무활동 비율·활동 지속기간 네 가지 관점에서 봅니다.

### 5-1. 총 클릭수(total_click) 비교

3-5에서 확인한 대로 sum_click은 극단치가 커서 total_click도 오른쪽으로 치우쳐 있을 수 있으므로,
평균과 중앙값을 함께 보고 분포는 로그 스케일로도 확인합니다.

In [ ]:
print(full_engagement.groupby("label_churn")["total_click"].agg(["mean", "median", "std", "count"]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=full_engagement, x="label_churn", y="total_click", ax=axes[0])
axes[0].set_title("완수/이탈 그룹별 총 클릭수")
axes[0].set_xlabel("이탈 여부")
axes[0].set_ylabel("총 클릭수")

# 극단치 때문에 원본 스케일에서는 분포가 잘 안 보이므로, log1p 변환한 값도 함께 확인
full_engagement["log_click"] = np.log1p(full_engagement["total_click"])
sns.boxplot(data=full_engagement, x="label_churn", y="log_click", ax=axes[1])
axes[1].set_title("완수/이탈 그룹별 총 클릭수 (로그 변환)")
axes[1].set_xlabel("이탈 여부")
axes[1].set_ylabel("log(1 + 총 클릭수)")

plt.tight_layout()
plt.show()

# [해석 포인트] 완수 그룹의 평균 클릭수(1,623)가 이탈 그룹(304)의 약 5.3배입니다.
# 로그 변환 후에도 두 그룹이 뚜렷하게 분리되어, 극단치 때문에 생긴 착시가 아님을 확인할 수 있습니다.

### 5-2. 활동일수(n_days_active) 비교

In [ ]:
print(full_engagement.groupby("label_churn")["n_days_active"].agg(["mean", "median", "std"]).round(1))

plt.figure(figsize=(6, 5))
sns.boxplot(data=full_engagement, x="label_churn", y="n_days_active")
plt.title("완수/이탈 그룹별 VLE 접속일수")
plt.xlabel("이탈 여부")
plt.ylabel("접속일수(일)")
plt.tight_layout()
plt.show()

# [해석 포인트] 완수 그룹은 평균 73.2일 접속하는데 이탈 그룹은 평균 15.8일(중앙값 6일)에 그칩니다.
# 중앙값 기준으로는 10배 넘게 차이가 나서, 총 클릭수보다도 더 뚜렷하게 갈립니다.

### 5-3. VLE 활동이 아예 없는 학생 비율 — 가장 강력한 신호

03번에서 "과제를 하나도 제출하지 않은 학생" 비율이 강력한 신호였던 것처럼, 여기서도 "VLE에
단 한 번도 접속하지 않은 학생" 비율을 확인합니다.

In [ ]:
zero_summary = full_engagement.groupby("label_churn").apply(
    lambda g: pd.Series({
        "무활동 인원": (g["n_days_active"] == 0).sum(),
        "전체 인원": len(g),
        "무활동 비율(%)": round((g["n_days_active"] == 0).mean() * 100, 1),
    }),
    include_groups=False,
)
print(zero_summary)

plt.figure(figsize=(6, 5))
zero_summary["무활동 비율(%)"].plot(kind="bar", color="salmon")
plt.title("완수/이탈 그룹별 VLE 무활동(접속 0일) 비율")
plt.ylabel("비율(%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**[확인 결과]** 완수 그룹은 VLE에 한 번도 접속하지 않은 학생이 **1.7%**뿐인데, 이탈 그룹은
**31.3%**에 달합니다. 이탈 학생 3명 중 1명은 온라인 강의실에 아예 들어와 본 적이 없다는
뜻입니다. 03번의 과제 제출률(보정 후 완수 15.8% vs 이탈 21.4%)보다도 훨씬 격차가 크고, 03번에서
지적했던 "탈퇴 시점 편향"도 4-1에서 이미 제거했으므로 이 수치는 순수하게 행동 차이를 반영합니다.

### 5-4. 활동 지속기간(active_span) 비교 (VLE 활동이 있었던 학생만)

활동이 아예 없는 학생은 active_span을 정의할 수 없으므로(4-3에서 NaN 처리), 여기서는 한 번
이상 접속한 학생만 대상으로 "학습이 얼마나 오래 이어졌는지"를 봅니다.

In [ ]:
active_only = full_engagement[full_engagement["n_days_active"] > 0]
print(active_only.groupby("label_churn")["active_span"].agg(["mean", "median", "std", "count"]).round(1))

plt.figure(figsize=(6, 5))
sns.boxplot(data=active_only, x="label_churn", y="active_span")
plt.title("완수/이탈 그룹별 활동 지속기간 (활동이 있었던 학생만)")
plt.xlabel("이탈 여부")
plt.ylabel("활동 지속기간(일)")
plt.tight_layout()
plt.show()

# [해석 포인트] 접속을 한 번이라도 한 학생끼리 비교해도, 완수 그룹은 평균 220일 동안 꾸준히
# 접속을 이어가는 반면 이탈 그룹은 평균 76일(중앙값 58일) 만에 접속이 끊깁니다.

### 5번 종합 결론

네 지표 모두 같은 방향을 가리킵니다.

| 지표 | 완수 | 이탈 | 배율/격차 |
|---|---|---|---|
| 총 클릭수 (평균/중앙값) | 1,623 / 999 | 304 / 74.5 | 약 5.3배 |
| 접속일수 (평균/중앙값) | 73.2일 / 62일 | 15.8일 / 6일 | 약 4.6배 |
| 무활동(0일 접속) 비율 | 1.7% | 31.3% | 약 18배 |
| 활동 지속기간 (접속자만, 평균) | 220.4일 | 76.0일 | 약 2.9배 |

가장 인상적인 건 **무활동 비율(1.7% vs 31.3%)**입니다. 03번에서 확인한 과제 제출률(탈퇴 시점
보정 후 15.8% vs 21.4%)보다 훨씬 격차가 크고, 여기서는 탈퇴 시점 편향도 4-1에서 이미 제거했기
때문에 순수한 행동 차이로 해석할 수 있습니다.

**[고용노동부 훈련과정 적용 시사점]** VLE 로그인/클릭 데이터는 과제 제출 데이터보다 훨씬 이른
시점부터, 그리고 더 빈번하게(거의 매일) 수집할 수 있는 지표입니다. "등록 후 첫 1~2주간 로그인
기록이 없음"을 조기경보 트리거로 삼으면, 03번에서 지적했던 "이탈 학생의 48.6%가 첫 과제 마감
전에 이미 이탈"하는 문제(과제 기반 지표로는 포착 불가능했던 조기 이탈군)까지 상당 부분 포착할
수 있을 것으로 보입니다.